# 07 — Robustness checks
Stress-tests the Family A vs. Family B comparison against three threats
identified in the audit:

1. **Outcome definition** — the primary target counts every
   non-employed respondent as "unemployed" (job-seekers, people not
   looking for work, homemakers, further study, occasional workers).
2. **Retrospective predictors** — the composite indices are rated in
   the same survey that reports employment, so they may be coloured by
   the outcome.
3. **Graduation cohort** — unemployment differs sharply by cohort, and
   cohorts may have been surveyed at different times after graduating.

**Protocol.** Every scenario uses repeated stratified 5-fold CV
(10 repeats) on *all available rows* (no 80/20 holdout), so scenarios
are comparable with each other. The reference scenario (primary target,
full feature set) is re-run under this same protocol; its numbers
differ slightly from notebooks 03–05, which use the 125-row training
split. All models are untuned, so no nested loop is needed.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import numpy as np
import pandas as pd
from IPython.display import display

from src.feature_engineering import (
    CANDIDATE_FEATURES, NUMERIC_FEATURES, NOMINAL_FEATURES, TARGET_COLUMN,
    PRE_OUTCOME_FEATURES, PRE_OUTCOME_NUMERIC, TARGET_VARIANTS, COHORT_COLUMN, POS_LABEL
)
from src.evaluation import (
    compare_models, pairwise_corrected_ttest, corrected_resampled_ttest,
    cv_split_sizes, N_SPLITS
)
from src.classification_models import build_baseline_models
from src.robustness import build_study_models, model_family, leave_one_group_out
from src.reporting import save_table

warnings.filterwarnings("ignore", category=UserWarning)

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tracer_study_processed.csv")
print(df.shape)

(157, 14)


## Scenarios

In [2]:
SCENARIOS = {
    "reference (primary target, all features)":
        dict(target=TARGET_COLUMN, features=CANDIDATE_FEATURES, numeric=NUMERIC_FEATURES),
    "labour-force target":
        dict(target="employment_status_labour_force", features=CANDIDATE_FEATURES, numeric=NUMERIC_FEATURES),
    "occasional work = employed":
        dict(target="employment_status_occasional_employed", features=CANDIDATE_FEATURES, numeric=NUMERIC_FEATURES),
    "pre-outcome features only":
        dict(target=TARGET_COLUMN, features=PRE_OUTCOME_FEATURES, numeric=PRE_OUTCOME_NUMERIC),
}

summaries, raw = {}, {}
for name, sc in SCENARIOS.items():
    data = df.dropna(subset=[sc["target"]])
    X, y = data[sc["features"]], data[sc["target"]].astype(int)
    models = build_study_models(sc["numeric"], NOMINAL_FEATURES, y_train=y)
    summary, scores = compare_models(models, X, y)
    summaries[name] = summary.assign(n=len(y), n_pos=int((y == POS_LABEL).sum()))
    raw[name] = (scores, len(y))
    print(f"{name}: n={len(y)}, at-risk={int((y == POS_LABEL).sum())}")

reference (primary target, all features): n=157, at-risk=31


labour-force target: n=145, at-risk=19


occasional work = employed: n=157, at-risk=26


pre-outcome features only: n=157, at-risk=31


## PR-AUC and ROC-AUC by scenario

`base_rate` is PR-AUC's chance level for that scenario; ROC-AUC's is 0.5.

In [3]:
cols = ["n", "n_pos", "base_rate", "pr_auc", "pr_auc_corrected_ci_low",
        "pr_auc_corrected_ci_high", "roc_auc"]
robustness_summary = pd.concat(
    {name: s[cols] for name, s in summaries.items()}, names=["scenario", "model"]
).reset_index()
robustness_summary["family"] = robustness_summary["model"].map(model_family)
save_table(robustness_summary, "robustness_summary", index=False)

pr_table = robustness_summary.pivot(index="model", columns="scenario", values="pr_auc")[list(SCENARIOS)]
roc_table = robustness_summary.pivot(index="model", columns="scenario", values="roc_auc")[list(SCENARIOS)]
print("PR-AUC"); display(pr_table.round(3))
print("ROC-AUC"); display(roc_table.round(3))

PR-AUC


scenario,"reference (primary target, all features)",labour-force target,occasional work = employed,pre-outcome features only
model,,,,
A1_RandomForest_ClassWeighted,0.302,0.273,0.230,0.251
A2_XGBoost_ScalePosWeight,0.266,0.225,0.196,0.267
A3_SMOTE_RandomForest,0.309,0.264,0.236,0.272
A4_ADASYN_RandomForest,0.301,0.298,0.245,0.254
B1_IsolationForest,0.275,0.241,0.250,0.235
B2_OneClassSVM,0.228,0.213,0.190,0.260
B3_LocalOutlierFactor,0.253,0.227,0.208,0.242
BL1_Dummy_Prior,0.197,0.131,0.166,0.197
BL2_LogReg_Balanced,0.361,0.315,0.296,0.387


ROC-AUC


scenario,"reference (primary target, all features)",labour-force target,occasional work = employed,pre-outcome features only
model,,,,
A1_RandomForest_ClassWeighted,0.552,0.581,0.511,0.534
A2_XGBoost_ScalePosWeight,0.487,0.493,0.426,0.550
A3_SMOTE_RandomForest,0.557,0.617,0.532,0.537
A4_ADASYN_RandomForest,0.543,0.615,0.526,0.527
B1_IsolationForest,0.532,0.588,0.555,0.452
B2_OneClassSVM,0.447,0.489,0.411,0.480
B3_LocalOutlierFactor,0.497,0.546,0.491,0.462
BL1_Dummy_Prior,0.500,0.500,0.500,0.500
BL2_LogReg_Balanced,0.638,0.631,0.620,0.665


## Does the Family A vs. B conclusion hold in each scenario?

Per scenario: how many of the 12 cross-family pairs favour Family A,
how many are significant (corrected resampled t-test, Bonferroni across
all 21 study-model pairs), and whether the best Family A model beats the
logistic-regression baseline (BL2), and whether BL2 beats the
prior-only dummy (BL1), i.e. whether the outcome is predictable at all.

In [4]:
rows = []
for name, (scores, n) in raw.items():
    n_train, n_test = cv_split_sizes(n, N_SPLITS)
    study = {k: v for k, v in scores.items() if model_family(k) in ("A", "B")}
    pairs = pairwise_corrected_ttest(study, n_train, n_test)
    cross = pairs[pairs["model_a"].map(model_family) != pairs["model_b"].map(model_family)]
    a_minus_b = np.where(cross["model_a"].map(model_family) == "A", 1, -1) * cross["mean_diff"]

    best_a = max((k for k in study if model_family(k) == "A"), key=lambda k: study[k].mean())
    _, p_vs_lr = corrected_resampled_ttest(scores[best_a], scores["BL2_LogReg_Balanced"], n_train, n_test)
    _, p_lr_vs_dummy = corrected_resampled_ttest(scores["BL2_LogReg_Balanced"], scores["BL1_Dummy_Prior"], n_train, n_test)

    rows.append({
        "scenario": name,
        "cross_pairs_favouring_A": int((a_minus_b > 0).sum()),
        "cross_pairs_significant": int(cross["significant"].sum()),
        "min_cross_p": cross["p_value"].min(),
        "best_A": best_a,
        "best_A_minus_LogReg": study[best_a].mean() - scores["BL2_LogReg_Balanced"].mean(),
        "best_A_vs_LogReg_p": p_vs_lr,
        "LogReg_minus_Dummy": scores["BL2_LogReg_Balanced"].mean() - scores["BL1_Dummy_Prior"].mean(),
        "LogReg_vs_Dummy_p": p_lr_vs_dummy,
    })

conclusion_check = pd.DataFrame(rows)
display(conclusion_check.round(4))
save_table(conclusion_check, "robustness_conclusion_check", index=False);

,scenario,cross_pairs_favouring_A,cross_pairs_significant,min_cross_p,best_A,best_A_minus_LogReg,best_A_vs_LogReg_p,LogReg_minus_Dummy,LogReg_vs_Dummy_p
0,"reference (primary target, all features)",11,0,0.2113,A3_SMOTE_RandomForest,-0.0517,0.4259,0.1631,0.0102
1,labour-force target,10,0,0.3864,A4_ADASYN_RandomForest,-0.0164,0.8489,0.1836,0.0157
2,occasional work = employed,7,0,0.3188,A4_ADASYN_RandomForest,-0.0508,0.3086,0.1306,0.0108
3,pre-outcome features only,10,0,0.4802,A3_SMOTE_RandomForest,-0.1145,0.0373,0.1892,0.0027


## Graduation cohort

Cohort (`batch`) is not a modeling feature, but if unemployment depends
strongly on cohort — e.g. because cohorts were surveyed at different
times after graduating — models may partly be learning cohort
differences. Three checks:

1. Unemployment rate by cohort.
2. A cohort-only baseline: logistic regression on cohort alone, same CV.
3. Leave-one-cohort-out: train on two cohorts, score the third.

In [5]:
by_cohort = (
    df.groupby(COHORT_COLUMN)[TARGET_COLUMN]
    .agg(n="size", n_unemployed=lambda s: int((s == POS_LABEL).sum()))
)
by_cohort["unemployment_rate"] = by_cohort["n_unemployed"] / by_cohort["n"]
display(by_cohort.round(3))
save_table(by_cohort, "cohort_unemployment")

X_cohort = df[[COHORT_COLUMN]].astype(str)
y = df[TARGET_COLUMN].astype(int)
cohort_only = {"BL3_LogReg_CohortOnly": build_baseline_models([], [COHORT_COLUMN])["BL2_LogReg_Balanced"]}
cohort_summary, _ = compare_models(cohort_only, X_cohort, y)
display(cohort_summary[["pr_auc", "pr_auc_corrected_ci_low", "pr_auc_corrected_ci_high",
                        "roc_auc", "base_rate"]].round(3))
save_table(cohort_summary, "cohort_only_baseline");

,n,n_unemployed,unemployment_rate
batch,,,
2023,61,16,0.262
2024,54,2,0.037
2025,42,13,0.310


,pr_auc,pr_auc_corrected_ci_low,pr_auc_corrected_ci_high,roc_auc,base_rate
BL3_LogReg_CohortOnly,0.278,0.225,0.33,0.648,0.197


In [6]:
X, y = df[CANDIDATE_FEATURES], df[TARGET_COLUMN].astype(int)
loco = leave_one_group_out(
    build_study_models(NUMERIC_FEATURES, NOMINAL_FEATURES, y_train=y), X, y, df[COHORT_COLUMN]
)
save_table(loco, "leave_one_cohort_out", index=False)

print("PR-AUC on the held-out cohort (base rate in column header):")
headers = {g: f"{g} (base {r:.2f})" for g, r in
           loco.groupby("held_out_group")["base_rate"].first().items()}
display(loco.pivot(index="model", columns="held_out_group", values="pr_auc")
        .rename(columns=headers).round(3))
print("ROC-AUC on the held-out cohort:")
display(loco.pivot(index="model", columns="held_out_group", values="roc_auc").round(3))

PR-AUC on the held-out cohort (base rate in column header):


held_out_group,2023 (base 0.26),2024 (base 0.04),2025 (base 0.31)
model,,,
A1_RandomForest_ClassWeighted,0.288,0.059,0.473
A2_XGBoost_ScalePosWeight,0.274,0.049,0.351
A3_SMOTE_RandomForest,0.280,0.143,0.512
A4_ADASYN_RandomForest,0.278,0.059,0.493
B1_IsolationForest,0.255,0.038,0.546
B2_OneClassSVM,0.248,0.053,0.343
B3_LocalOutlierFactor,0.236,0.041,0.362
BL1_Dummy_Prior,0.262,0.037,0.310
BL2_LogReg_Balanced,0.312,0.085,0.554


ROC-AUC on the held-out cohort:


held_out_group,2023,2024,2025
model,,,
A1_RandomForest_ClassWeighted,0.507,0.562,0.642
A2_XGBoost_ScalePosWeight,0.452,0.452,0.565
A3_SMOTE_RandomForest,0.497,0.760,0.716
A4_ADASYN_RandomForest,0.497,0.548,0.698
B1_IsolationForest,0.478,0.279,0.613
B2_OneClassSVM,0.465,0.385,0.406
B3_LocalOutlierFactor,0.426,0.298,0.501
BL1_Dummy_Prior,0.500,0.500,0.500
BL2_LogReg_Balanced,0.600,0.692,0.695


## Split sensitivity

Notebooks 03–05 cross-validate on one 125-row training split
(`train_test_split(..., random_state=42)`) and score the 32-row holdout
once. This repeats that exact protocol for four other split seeds, to
show how much the headline ranking depends on which 125 rows were drawn.
`cv_pr_auc` is the 5x10 CV mean on the training split; `holdout_pr_auc`
is the model refit on that split and scored on its 32-row holdout
(6 at-risk graduates).

In [7]:
from sklearn.model_selection import train_test_split
from src.evaluation import pr_auc_scorer

SPLIT_SEEDS = [42, 0, 1, 2, 3]
X_all, y_all = df[CANDIDATE_FEATURES], df[TARGET_COLUMN].astype(int)

rows = []
for seed in SPLIT_SEEDS:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.2, random_state=seed, stratify=y_all
    )
    models = build_study_models(NUMERIC_FEATURES, NOMINAL_FEATURES, y_train=y_tr)
    summary, _ = compare_models(models, X_tr, y_tr)
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        rows.append({"split_seed": seed, "model": name, "family": model_family(name),
                     "cv_pr_auc": summary.loc[name, "pr_auc"],
                     "holdout_pr_auc": pr_auc_scorer(model, X_te, y_te)})

split_sensitivity = pd.DataFrame(rows)
save_table(split_sensitivity, "split_sensitivity", index=False)

print("CV PR-AUC on the 125-row training split, by split seed (42 = notebooks 03-05):")
display(split_sensitivity.pivot(index="model", columns="split_seed", values="cv_pr_auc")
        [SPLIT_SEEDS].round(3))
print("Holdout PR-AUC (32 rows, 6 at-risk), by split seed:")
display(split_sensitivity.pivot(index="model", columns="split_seed", values="holdout_pr_auc")
        [SPLIT_SEEDS].round(3))

CV PR-AUC on the 125-row training split, by split seed (42 = notebooks 03-05):


split_seed,42,0,1,2,3
model,,,,,
A1_RandomForest_ClassWeighted,0.416,0.362,0.310,0.283,0.250
A2_XGBoost_ScalePosWeight,0.386,0.328,0.274,0.273,0.211
A3_SMOTE_RandomForest,0.399,0.368,0.331,0.291,0.283
A4_ADASYN_RandomForest,0.436,0.338,0.301,0.271,0.263
B1_IsolationForest,0.282,0.317,0.299,0.314,0.294
B2_OneClassSVM,0.207,0.260,0.231,0.268,0.252
B3_LocalOutlierFactor,0.241,0.293,0.257,0.298,0.281
BL1_Dummy_Prior,0.200,0.200,0.200,0.200,0.200
BL2_LogReg_Balanced,0.343,0.408,0.345,0.343,0.299


Holdout PR-AUC (32 rows, 6 at-risk), by split seed:


split_seed,42,0,1,2,3
model,,,,,
A1_RandomForest_ClassWeighted,0.149,0.224,0.372,0.261,0.625
A2_XGBoost_ScalePosWeight,0.156,0.214,0.392,0.231,0.671
A3_SMOTE_RandomForest,0.144,0.189,0.314,0.287,0.488
A4_ADASYN_RandomForest,0.145,0.185,0.291,0.338,0.544
B1_IsolationForest,0.339,0.273,0.211,0.188,0.264
B2_OneClassSVM,0.459,0.165,0.179,0.190,0.203
B3_LocalOutlierFactor,0.390,0.157,0.216,0.187,0.220
BL1_Dummy_Prior,0.188,0.188,0.188,0.188,0.188
BL2_LogReg_Balanced,0.442,0.230,0.480,0.290,0.518
